In [1]:
language = 'pt'

# 1. Gravação de Áudio Com Python (e Uma Pitada de JavaScript) 🎤

In [2]:
# Referência: https://gist.github.com/korakot/c21c3476c024ad6d56d5f48b0bca92be

from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

# Código JavaScript para gravar áudio do usuário usando a "MediaStream Recording API"
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
  # Executa o código JavaScript para gravar o áudio
  display(Javascript(RECORD))
  # Recebe o áudio gravado como resultado do JavaScript
  js_result = output.eval_js('record(%s)' % (sec * 1000))
   # Decodifica o áudio em base64
  audio = b64decode(js_result.split(',')[1])
  # Salva o áudio em um arquivo
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  # Retorna o caminho do arquivo de áudio (pasta padrão do Google Colab)
  return f'/content/{file_name}'

# Grava o áudio do usuário por um tempo determinado (padrão 5 segundos)
print('Ouvindo...\n')
record_file = record()

# Exibe o áudio gravado
display(Audio(record_file, autoplay=False))

Ouvindo...



<IPython.core.display.Javascript object>

# 2. Reconhecimento de Fala com Whisper (OpenAI) 🧠

In [3]:
!pip install git+https://github.com/openai/whisper.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
import whisper

# Selecione o modelo do Whisper que melhor atenda às suas necessidades:
# https://github.com/openai/whisper#available-models-and-languages
model = whisper.load_model("small")

# Transcreve o audio gravado anteriormente.
result = model.transcribe(record_file, fp16=False, language=language)
transcription = result["text"]
print(transcription)

 Quais são os cuidados da unidilização de medicamentos termoláveis?


# 3. Integração com a API do Gemini 💬

In [7]:
!pip install -q -U google-generativeai

In [20]:
import google.generativeai as genai
import os

GOOGLE_API_KEY = "SUA-API-KEY"
genai.configure(api_key=GOOGLE_API_KEY)

model = genai.GenerativeModel('models/gemini-3.1-flash-lite-preview')

prompt_hospitalar = f"""
Você é um Auxiliar de Farmácia Hospitalar.
Sua função é ajudar em rotinas de unitarização de doses, conferência de carrinhos de emergência,
verificação de validades e organização de estoque hospitalar.
Responda de forma rápida, técnica e prática à seguinte solicitação: {transcription}
"""

response = model.generate_content(prompt_hospitalar)
gemini_response = response.text

print("Resposta do Assistente Farmacêutico:")
print(gemini_response)

Resposta do Assistente Farmacêutico:
Para a unitarização de termoláveis (2°C a 8°C), siga estes cuidados críticos:

1.  **Redução do Tempo de Exposição:** Mantenha os itens fora do refrigerador pelo tempo mínimo necessário. Utilize **caixas térmicas validadas** com gelox durante o processo de fracionamento.
2.  **Manutenção da Cadeia de Frio:** Monitore a temperatura da área de trabalho. Se a umidade ou temperatura excederem os limites do protocolo institucional, interrompa a manipulação imediatamente.
3.  **Integridade da Embalagem Primária:** Não remova o medicamento do seu invólucro original ou proteção contra luz (se houver) até o momento da unitarização. 
4.  **Identificação (Rotulagem):** O rótulo deve conter obrigatoriamente: nome do fármaco, concentração, lote, validade original e **a nova data de validade reduzida** (se aplicável após abertura/fracionamento), conforme POP da farmácia.
5.  **Acondicionamento pós-unitarização:** Após o recondicionamento, retorne o medicamento im

# 4. Sintetizando a Resposta do Gemini Como Voz (gTTS) 🔊

In [13]:
!pip install gTTS

In [21]:
from gtts import  gTTS

# Cria um objeto gTTS com a resposta gerada pelo Gemini e a língua que será sintetizada em voz (variável "language").
gtts_object = gTTS(text=gemini_response, lang=language, slow=False)

# Salva o áudio da resposta no arquivo especificado (pasta padrão do Google Colab)
response_audio = "/content/response_audio.wav"
gtts_object.save(response_audio)

# Reproduz o áudio da resposta salvo no arquivo
display(Audio(response_audio, autoplay=True))